# 11.5 多開一些文字層，怎麼比較成本與效果？


圖片問答可以只調接頭，也可以連最後文字區塊一起調，或開放整個模型。先數每種方案允許改幾個數字，再用相同資料與預算比較答案；參數多不代表新題一定更好。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
from tiny_perceptron.model import TinyLM, ModelConfig
from tiny_perceptron.multimodal import MultiModalLM

model = MultiModalLM(TinyLM(ModelConfig(width=8)))
for mode in ["projector", "partial", "all"]:
    model.requires_grad_(mode == "all")
    model.image_projector.requires_grad_(True)
    if mode == "partial":
        model.language.blocks[-1].requires_grad_(True)
    count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(mode, count)

每輪先重設開關，再開所需部分，避免上輪殘留。寬 8 的預設模型印 projector 136、partial 976、all 8296。partial 在這段程式只開最後一個文字區塊與接頭；all 包含聲音零件，純圖片輸入未必讓它們收到梯度。

可訓練數量是成本線索。參數本身之外，訓練還可能存梯度與 AdamW 歷史狀態；實際用量取決於參數是否參與更新。新任務效果則要從相同起點、同樣題目與有效回答位置比較，並另測原文字任務。

[既有小對照](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/vqa.json)真正訓練的是文字寬 64、兩層，圖片入口寬 16 的模型，與上面的寬 8 參數計數例不同。材料是紅、綠、藍各配圓形、方形的六種合成圖，每圖各問顏色與形狀，答案分別是 `red`／`green`／`blue` 與 `circle`／`square`；同一原圖的兩題保持同組。訓練圖的圖形中心向左移 2 或 1 像素，或不移；驗證向右移 1，最後檢查向右移 2，因此留出的是新位置，顏色與形狀組合仍是已知的六種。

每版都使用3,830個有效回答目標；驗證十二題與最後檢查十二題是分開的兩組題，結果如下：

| 更新範圍 | 可訓練參數 | 驗證答對 | 最後題答對 |
| --- | ---: | ---: | ---: |
| 圖片接頭 | 1,088 | 3/12 | 3/12 |
| 接頭＋最後文字區塊 | 50,816 | 10/12 | 12/12 |
| 全部 | 145,664 | 12/12 | 9/12 |

最多參數沒有最高最後題成績；中間方案的驗證也有錯題。這批只有十二題，不能選成所有模型的最佳範圍。比較的是這份資料與配方，而不是單看參數排名。

練習將短程式的 `ModelConfig(width=8)` 改成 `ModelConfig(width=8, layers=2)`，接頭仍 136，partial 仍只開最後一層，all 多一個區塊。寬度、深度與更新範圍是三種選擇。重做表中實驗時，固定工具的 partial 是最後區塊；[T.6的訓練CLI](https://birdhackor.github.io/tiny-perceptron-vlm/training.html#T.6)則把partial定義為接頭加首末文字區塊，按所用入口的約定選擇更新範圍。

<details>
<summary>回顧與查證</summary>

可回顧：[11.2參數凍結與優化器](https://birdhackor.github.io/tiny-perceptron-vlm/11.2.html)、[4.5一個模型區塊](https://birdhackor.github.io/tiny-perceptron-vlm/4.5.html)。

原實驗的資料切分、更新設定與逐題結果見[既有實報](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/vqa.json)；重做入口見[實驗說明](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/README.md)。這是上述有限任務的歷史紀錄。

</details>
